# DM-AHR-ViT — run the pipeline natively on a local GPU or Colab GPU

Every cell here runs the study's own scripts directly with `python`. **No Modal account, no token,
no `modal run`.** The GPU is whatever machine the notebook is on.

Use this notebook if you want to see the pipeline execute yourself, on your own hardware.

**Research prototype only. Not clinically validated, not for diagnosis or patient care.**

| | |
|---|---|
| Works on | local NVIDIA GPU, Colab (T4 / L4 / A100), any CUDA box |
| Needs | ~4 GB free disk for raw data, ~2 GB for processed arms |
| Does not need | Modal, Kaggle credentials, DERMAHAIR (that path is optional and blocked) |
| Time | ~8–14 GPU-hours for the full study; phases 1–2 are minutes |

**What has been verified.** Phases 1–3 (download, audit and split, hair pipeline) were executed
natively on CPU with no Modal present, from a fresh download: they reproduced the reference split
hash `fda56cf3…`, the same 7,470 lesion groups, the same 8,993 hair-present count and the same 1,022
synthetic pairs. Phases 4–11 need a CUDA GPU and were **not** executed during that verification; the
commands are the same ones the Modal runs used.

Read section 11 before starting a full run. Arm C sampling dominates the cost and is the phase that
most often outlives a Colab session.

## 1. Check the environment

The scripts pick their device themselves: `cuda` if `torch.cuda.is_available()`, otherwise CPU.
Nothing else needs configuring.

In [ ]:
import subprocess, sys, os

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout or
      "no nvidia-smi on PATH (fine if torch still sees a GPU below)")

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0))
        print("bf16 supported:", torch.cuda.is_bf16_supported())
except ImportError:
    print("torch not installed yet - run the next cell")

## 2. Install dependencies

`requirements.txt` pins the exact versions the reported results were produced with. On Colab this
takes a couple of minutes and will replace Colab's preinstalled torch — that is intended, because a
different torch version is a different experiment.

`modal` is listed there only because the project supports running on Modal; installing it is
harmless and it is never imported by the pipeline code.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("dependencies installed")

## 3. Get the code and choose where data lives

**In Colab, point `DATA_ROOT` and `ARTIFACT_ROOT` at Drive.** `/content` is wiped when the session
ends, and Arm C alone is hours of work. A local machine can use paths inside the repo.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/supraehhe-web/ML-shit.git"
BRANCH = "hoplite/dm-ahr-vit-pipeline"

if not Path("configs/protocol.json").exists():
    !git clone --branch {BRANCH} {REPO_URL} ML-shit
    %cd ML-shit

print("repository:", Path.cwd())

In [ ]:
import os
from pathlib import Path

# --- Colab: put data on Drive so it survives the session ---
IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/dmahr-vit")
else:
    ROOT = Path.cwd() / "runtime"

DATA_ROOT = ROOT / "data"
ARTIFACT_ROOT = ROOT / "artifacts"
for p in (DATA_ROOT, ARTIFACT_ROOT):
    p.mkdir(parents=True, exist_ok=True)

os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("ARTIFACT_ROOT=", ARTIFACT_ROOT)

In [ ]:
# A tiny wrapper so every later cell runs a project script with those env vars.
import subprocess, sys, os
from pathlib import Path

def run(script, *args, check=True):
    cmd = [sys.executable, str(Path("scripts") / script), *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, env=os.environ.copy(), check=check).returncode

print("helper ready")

## 4. Phase 1 — download the data

Downloads the client's ISIC collection 212 (all 11,720 images) and the HAM10000 metadata plus
segmentations from Harvard Dataverse. **All public: no account, no key.** Takes a few minutes.

Measured on a CPU-only machine: 11,720 images plus 10,016 masks, 339 MB, **6.6 minutes**, no GPU.

`--limit N` exists for smoke tests, but a limited download cannot produce the real cohort — phase 2
will refuse to freeze a split that does not match the expected counts. Do not mix them.

In [ ]:
run("00_download_data.py")

In [ ]:
import json, os
from pathlib import Path
rep = json.loads((Path(os.environ["ARTIFACT_ROOT"]) / "logs" / "download_report.json").read_text()) \
      if (Path(os.environ["ARTIFACT_ROOT"]) / "logs" / "download_report.json").exists() else {}
print(json.dumps({k: rep[k] for k in ("isic", "masks_extracted") if k in rep}, indent=2)[:800])

## 5. Phase 2 — audit, deduplicate and freeze the split

Recomputes every count from the files, merges pixel-confirmed near-duplicates into lesion groups,
and builds the class-stratified 70/15/15 split with zero group overlap. The split hash is frozen, so
any later change is detectable.

This is CPU-only and takes a few minutes. Run it once and keep the manifests — everything downstream
depends on them, and re-running it after Arm C exists would silently change what the checkpoints were
trained on.

Measured: **83 seconds** on CPU. From a completely fresh download this reproduces the reference split
hash `fda56cf3…` byte for byte, so the split is deterministic across machines.

In [ ]:
run("01_audit_and_splits.py")

In [ ]:
import json, os
from pathlib import Path
log = Path(os.environ["ARTIFACT_ROOT"]) / "logs"
audit = json.loads((log / "data_audit.json").read_text())
print("collection:", audit["collection_total"])
print("confirmatory C0:", audit["c0_decoded"], "| quarantined extras:", audit["c0_extra"])
print("lesion groups:", audit["groups_total"])
print("frozen split:", audit["frozen_splits"]["counts"].keys() if "counts" in audit["frozen_splits"] else "")
print("leakage check:", audit.get("split_verification"))

## 6. Phase 3 — hair screening, Arm B and synthetic pairs

Detects hair, produces the classical Arm B images, and composites synthetic hairy/hairless pairs.

The screening threshold is frozen here, before any test metric exists. This is CPU work and takes
roughly 10–20 minutes over 10k images.

Measured: **10 minutes** on CPU (531 s for the removal, 69 s for the synthetic pairs). Produces 8,993
hair-present images, 10,015 Arm B and 10,015 Arm A\* images, and 1,022 synthetic pairs (654 train /
182 val / 186 test) — identical to the reference run, with the same config hash.

In [ ]:
run("02_hair_pipeline.py")

## 7. Phase 4 — DM-AHR smoke test

Needs the authors' released checkpoint, which is **not** in the repository. It comes from the
DM-AHR Google Drive folder and lands in `$DATA_ROOT/raw/dmahr_weights/`.

The smoke test is not optional: it establishes that the vendored upstream code runs, that the two
upstream defects are worked around correctly, and that a batch round-trips deterministically. Skip
it and Arm C may silently produce garbage.

If the download fails, Arm C is still possible — phase 6 trains a checkpoint from scratch — but you
then have no external sanity check on the adapter.

In [ ]:
from pathlib import Path
import os

weights_dir = Path(os.environ["DATA_ROOT"]) / "raw" / "dmahr_weights"
weights_dir.mkdir(parents=True, exist_ok=True)
expected = weights_dir / "I725000_E1083_gen.pth"

if expected.exists():
    print("already present:", expected)
else:
    !gdown --folder "https://drive.google.com/drive/folders/1OINKjCZQNzU1OxFQ64i3GlYkgbe_zrIh" -O {weights_dir}
    print("\nfiles now present:")
    print(sorted(p.name for p in weights_dir.rglob("*.pth")))

In [ ]:
run("03_dmahr_smoke.py")

In [ ]:
# Throughput benchmark on this GPU. Use it to decide whether the full Arm C run fits your budget.
run("03b_dmahr_bench.py", check=False)

## 8. Phase 5 — train the diffusion model

Trained on train split sources only, selected on validation. The released weights are used for the
smoke test only, because their training data is undocumented.

**This is the long one on a small GPU.** `configs/dmahr.json` controls iterations; the reported run
used 20,000 against 660,000 upstream. It checkpoints as it goes, so an interrupted run resumes.

On a T4, expect this to be slow enough that you should reduce iterations for a first pass.

In [ ]:
# quick sanity pass first: overfit a handful of pairs and confirm the loss collapses
run("04_dmahr_train.py", "--overfit", check=False)

In [ ]:
run("04_dmahr_train.py")

## 9. Phase 6 — generate Arm C

Applies the trained checkpoint to every hair-present image, then to the held-out test pairs.

Without Modal there is no automatic spawning, so the shards are just a loop. Sharding still matters:
each shard skips images already written, so a killed Colab session resumes instead of restarting.

`--num-timesteps 1000` reproduces the reported ablation at about half the cost. Drop the flag for
the 2,000-step reference settings. Every per-image row records `num_timesteps`.

**Start with a small shard count and check the timing before committing to all of it.**

In [ ]:
import time
from pathlib import Path
import os

# One shard first, to measure throughput on this GPU before paying for all of it.
t0 = time.time()
run("05_dmahr_sample.py", "--shard", 0, "--n-shards", 8, "--bf16", "--num-timesteps", 1000)
print(f"shard 0 of 8 took {(time.time() - t0) / 60:.1f} min")
print("If that is acceptable, run the remaining shards in the next cell.")

In [ ]:
# Remaining shards. Safe to re-run: finished images are skipped.
for shard in range(1, 8):
    run("05_dmahr_sample.py", "--shard", shard, "--n-shards", 8, "--bf16", "--num-timesteps", 1000)

In [ ]:
# Held-out test pairs, needed for PSNR/SSIM.
run("05_dmahr_sample.py", "--pairs-split", "test", "--bf16", "--num-timesteps", 1000)

In [ ]:
# Hair-absent images are identical in every arm; copy them into Arm C.
run("05b_fill_c_passthrough.py")

## 10. Phase 7 — classic and control arms

Arm A needs no processing (the dataset reads the original images). Arm A\* is the 128 px
round-trip that controls for Arm C's resolution change. `02_hair_pipeline.py` wrote A\* alongside B,
so there is usually nothing to do here — this cell just reports what exists.

In [ ]:
from pathlib import Path
import os

processed = Path(os.environ["DATA_ROOT"]) / "processed"
for arm in ("A", "B", "C", "Astar"):
    d = processed / arm
    n = len(list(d.rglob("*.jpg"))) + len(list(d.rglob("*.png"))) if d.exists() else 0
    print(f"{arm:6s} images on disk: {n}")
print("\nA reads the raw images directly; B and Astar come from 02_hair_pipeline.py.")

## 11. Phase 8 — train the ViT classifiers

Four arms × three seeds. Each run is resumable and writes its own checkpoint directory with the
config, seed, class weights and git commit.

Measured on an A100: about 17 s per epoch and roughly 25 epochs to early stopping, so ~7 minutes per
run, ~1.5 hours for all twelve. On a T4 allow several times that.

**Run one first.** If a single run works and the validation curve looks sane, launch the rest.

In [ ]:
# One run, to confirm the recipe works on this machine and data.
run("06_train_vit.py", "--arm", "A", "--seed", 42)

In [ ]:
import time

for arm in ["A", "Astar", "B", "C"]:
    for seed in [42, 123, 2026]:
        if (arm, seed) == ("A", 42):
            continue  # already trained above
        t0 = time.time()
        print(f"\n=== arm {arm} seed {seed} ===", flush=True)
        run("06_train_vit.py", "--arm", arm, "--seed", seed)
        print(f"finished in {(time.time() - t0) / 60:.1f} min", flush=True)

## 12. Phase 9 — evaluate on the locked test set

Scored once per frozen checkpoint, producing per-image predictions rather than only aggregates, so
every downstream statistic can be recomputed.

In [ ]:
for arm in ["A", "Astar", "B", "C"]:
    for seed in [42, 123, 2026]:
        run("07_eval_vit.py", "--arm", arm, "--seed", seed)

## 13. Phase 10 — image quality and explanations

PSNR/SSIM against the synthetic hairless references, then Grad-CAM and attention roll-out against
the lesion masks. XAI is the slower of the two: it runs a forward and backward pass per test image.

In [ ]:
run("07b_image_quality.py", "--split", "test")

In [ ]:
for arm in ["A", "Astar", "B", "C"]:
    for seed in [42, 123, 2026]:
        run("08_xai.py", "--arm", arm, "--seed", seed)

## 14. Phase 11 — statistics, report and gates

Bootstrap plus Wilcoxon, both Holm-corrected; the report generated from artifacts; the artifact
inventory with SHA-256 hashes; and the acceptance check.

The acceptance gates will disagree with the reference run in one respect: `03b_dmahr_bench.py` may
not have produced measurements on your GPU. Gates that check for a *specific* artifact will fail
honestly rather than pass silently — read the failures rather than assuming the pipeline broke.

In [ ]:
run("09_stats.py")
run("10_finalize.py")
run("check_acceptance.py", check=False)

In [ ]:
import pandas as pd, os
from pathlib import Path

logs = Path(os.environ["ARTIFACT_ROOT"]) / "logs"
stats = pd.read_csv(logs / "statistical_comparisons.csv")
print(f"{len(stats)} comparisons across {stats.endpoint.nunique()} endpoints\n")
print(stats[["endpoint", "subset", "comparison", "difference", "ci_low", "ci_high",
             "p_holm", "wilcoxon_p_holm"]].round(4).to_string(index=False))

## 15. Where the results are

Everything is under `ARTIFACT_ROOT`. To explore the numbers interactively — hypothesis verdicts,
figures, acceptance gates — point `01_results_walkthrough.ipynb` at the same tree by setting the same
two environment variables before running it.

In [ ]:
import os
from pathlib import Path

root = Path(os.environ["ARTIFACT_ROOT"])
for sub in ("logs", "predictions", "panels", "reports", "checkpoints"):
    d = root / sub
    n = sum(1 for _ in d.rglob("*")) if d.exists() else 0
    print(f"{sub:14s} {n:5d} files  {d}")

## 16. Practical notes

**Persistence.** In Colab, everything outside Drive is lost when the session ends. Set `ROOT` to
Drive in section 3 and every phase becomes resumable across sessions.

**Cost.** The whole study is 8–14 A100-hours, dominated by Arm C sampling over 8,993 images. A T4 is
several times slower per image, so the same run can span many sessions. Phases 2, 3 and 11 are CPU;
phases 4–10 need the GPU.

**Resumability.** Diffusion training checkpoints as it goes, Arm C skips images already written, and
ViT training writes per-epoch state. Re-running a phase is safe; it continues rather than restarts.

**Changing the split.** Do not re-run phase 2 after training. New manifests would not match the
checkpoints, and every comparison downstream would be invalid.

**What will differ from the reference run.** Absolute timings, and therefore any benchmark-derived
numbers, depend on the GPU. The scientific results should reproduce exactly if the seeds, data and
configs are unchanged; the reference run confirmed this for Arm A seed 42 to six decimal places.